## Session 3: Using gene descriptions and GPT to generate gene embeddings
(*updated to 2026 for OpenRouter API*)

---

기존 Foundation Model은 유전자 표현 Profile에서 유전자 및 세포 기능을 간접 학습합니다. 이 때문에, 광범위한 데이터 큐레이션 및 많은 리소스 투입을 요구합니다.

반면, 개별 유전자의 NCBI text description 및 GPT를 활용하여 gene embedding을 생성하고, 이 gene embedding으로 각종 task를 수행합니다.

In [ ]:
from openai import OpenAI

import pandas as pd
import json 
import numpy as np
import pickle

import sklearn
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

In [ ]:
from pathlib import Path

# Download the tutorial data into the Colab runtime.
# -nc prevents repeat runs from creating duplicate files with .1 suffixes.
PATH = Path("data")
OPENROUTER_API_KEY = ""

BASE_URL = "https://raw.githubusercontent.com/leelabsg/clinical-llm-tutorial/main/Session3/data"

PATH.mkdir(parents=True, exist_ok=True)
(PATH / "gene2vec").mkdir(exist_ok=True)

!wget -nc -q -P $PATH $BASE_URL/gene_embeddings_text-embedding-3-small_trimmed.pkl
!wget -nc -q -P $PATH $BASE_URL/NCBI_cleaned_summary_of_genes.json
!wget -nc -q -P $PATH/gene2vec $BASE_URL/gene2vec/test_label.txt
!wget -nc -q -P $PATH/gene2vec $BASE_URL/gene2vec/test_text.txt
!wget -nc -q -P $PATH/gene2vec $BASE_URL/gene2vec/train_label.txt
!wget -nc -q -P $PATH/gene2vec $BASE_URL/gene2vec/train_text.txt

print(f"Data ready in: {PATH.resolve()}")

## 1. Set Up OpenRouter API to use embeddings

In [ ]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# ── Privacy provider config (reused across all requests) ──────
PROVIDER_CONFIG = {
    "zdr": True,
    "data_collection": "deny",
}

Embedding 모델은 OpenAI의 "text-embedding-3-small" 사용, 용도에 따라서 다른 Embedding Model로도 대체 가능

In [ ]:
EMBED_MODEL = "openai/text-embedding-3-small"
BATCH_SIZE = 256

In [ ]:
## 전 단계에서 생성한 NCBI gene summary 파일 적재
with open(PATH / "NCBI_cleaned_summary_of_genes.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)



In [ ]:
print(summaries['CCND1'])

In [ ]:
text_input = summaries['CCND1']
resp = client.embeddings.create(model=EMBED_MODEL, input=text_input)
print(resp.data[0].embedding)
print(len(resp.data[0].embedding))

### 2. Utilize generated gene embeddings in tasks

In [ ]:
import pickle

with open(PATH / "gene_embeddings_text-embedding-3-small_trimmed.pkl", "rb") as fp:
    GPT_gene_embeddings = pickle.load(fp)

In [ ]:
print(GPT_gene_embeddings['CCND1'])
len(GPT_gene_embeddings)

#### [Task] Gene-gene interaction prediction

본 예제에서는 생성된 gene embedding의 gene-gene interaction 예측 성능을 Gene2vec 및 무작위 embeddings에 비교해서 보여줍니다.

The input data downloaded from https://github.com/jingcheng-du/Gene2vec repo.

In [ ]:
# gene gene interaction 
train_text_GGI = pd.read_csv(PATH / "gene2vec" / "train_text.txt", sep=' ', header=None)
train_label_GGI = pd.read_csv(PATH / "gene2vec" / "train_label.txt", header=None)
train_text_GGI.columns = ['gene_1','gene_2']
train_label_GGI.columns = ['label']
train_text_GGI_df = pd.concat([train_text_GGI,train_label_GGI], axis=1)

test_text_GGI = pd.read_csv(PATH / "gene2vec" / "test_text.txt", sep=' ', header=None)
test_label_GGI = pd.read_csv(PATH / "gene2vec" / "test_label.txt", header=None)
test_text_GGI.columns = ['gene_1','gene_2']
test_label_GGI.columns = ['label']
test_text_GGI_df = pd.concat([test_text_GGI,test_label_GGI], axis=1)

In [ ]:
print(train_text_GGI_df.head())


Featurization applied by adding embedding vectors of the two genes together

1. Gene-gene interaction with gene embeddings

In [ ]:
X_array_train = []
y_array_train = []
for i, row in train_text_GGI_df.iterrows():
    if row['gene_1'] in  GPT_gene_embeddings \
        and row['gene_2'] in GPT_gene_embeddings:
        X_array_train.append(GPT_gene_embeddings[row['gene_1']]+\
                             GPT_gene_embeddings[row['gene_2']])
        y_array_train.append(row['label'])
X_array_train = np.array(X_array_train)
y_array_train = np.array(y_array_train)
X_array_train = X_array_train.reshape(X_array_train.shape[0],-1)

X_array_test = []
y_array_test = []
for i, row in test_text_GGI_df.iterrows():
    if row['gene_1'] in  GPT_gene_embeddings \
        and row['gene_2'] in GPT_gene_embeddings:
        X_array_test.append(GPT_gene_embeddings[row['gene_1']]+\
                             GPT_gene_embeddings[row['gene_2']])
        y_array_test.append(row['label'])
X_array_test = np.array(X_array_test)
y_array_test = np.array(y_array_test)
X_array_test = X_array_test.reshape(X_array_test.shape[0],-1)

In [ ]:
print(X_array_train.shape, y_array_train.shape, X_array_test.shape, y_array_test.shape)
print(y_array_train[:5,])

In [ ]:
# Using logistic regression for the gene embeddings
lr = LogisticRegression()  # Add hyperparameters like n_estimators if needed
lr.fit(X_array_train, y_array_train)
y_pred_lr = lr.predict_proba(X_array_test)

# Computing the ROC curve with _lr suffix
fpr_lr, tpr_lr, _ = sklearn.metrics.roc_curve(y_array_test, y_pred_lr[:, 1])


In [ ]:
# Plot ROC curves
plt.figure()
# Existing Logistic Regression curves
# ... your existing plot code for Logistic Regression here ...
plt.figure(figsize=(6,5))
# Add Random Forest ROC curves
plt.plot(fpr_lr, tpr_lr, color='blue', lw=2, 
         label='Gene embeddings (AUC = %0.2f)' % (sklearn.metrics.roc_auc_score(y_array_test, y_pred_lr[:,1])))

# plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC for gene-gene interaction prediction with a random forest classifier')
plt.legend(loc='upper left',bbox_to_anchor=(-0.3, -0.15),fontsize=11,ncol=3)